# Radiology Reporting Harness — pipeline notebook

Self-contained. Built from the project `src/` modules by `notebook/build_notebook.py` — edit there, not here. Scorer, retrieval, prompt and post-processor are identical to the locally-tested versions.

In [ ]:
# ===== scorer (res_score.py) =====
"""
Local re-implementation of the competition's Radiology Edit Score (RES).

Spec: ../EVALUATION.md  (verbatim from the Kaggle Evaluation tab).

This is a best-effort reconstruction. Constants that the host did not fully specify
are collected in `Config` and must be calibrated against the first real leaderboard
result. The goal is high *rank correlation* with the hidden scorer, not identity.

Usage:
    from res_score import score_case, leaderboard_res
    r = score_case(pred_report, ref_report, template_content)   # -> dict, r["res"]
    lb = leaderboard_res([(pred, ref, tmpl), ...])              # -> float (mean RES)
"""
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass, field
from functools import lru_cache
from typing import Dict, List, Tuple

# --------------------------------------------------------------------------------------
# Token classification
# --------------------------------------------------------------------------------------

NEGATION = {
    "no", "not", "without", "negative", "absent", "none", "nor", "neither", "non",
    "unremarkable", "normal", "clear", "patent", "intact", "unchanged",
}
LATERALITY = {
    "left", "right", "bilateral", "bilaterally", "unilateral", "lt", "rt", "bilat",
    "midline", "central", "medial", "lateral", "proximal", "distal", "superior",
    "inferior", "anterior", "posterior",
}
SEVERITY = {
    "mild", "mildly", "moderate", "moderately", "severe", "severely", "minimal",
    "minimally", "marked", "markedly", "trace", "small", "large", "tiny", "extensive",
    "significant", "gross", "subtle", "prominent", "borderline", "slight", "slightly",
    "grade", "low", "high",
}
ACUITY = {
    "acute", "chronic", "subacute", "old", "new", "healing", "healed", "interval",
    "age", "aged", "remote", "recent", "progressive", "stable",
}
UNITS = {"mm", "cm", "ml", "cc", "mmhg", "hu", "cm2", "mm2", "mm3", "cm3"}

CRITICAL_WORDS = NEGATION | LATERALITY | SEVERITY | ACUITY | UNITS

FUNCTION_WORDS = {
    "the", "a", "an", "and", "or", "of", "with", "is", "are", "was", "were", "be",
    "been", "being", "to", "in", "on", "at", "as", "by", "for", "that", "this",
    "these", "those", "there", "it", "its", "from", "than", "then", "which", "has",
    "have", "had", "do", "does", "did", "into", "within", "also", "but", "if",
}

_NUM_RE = re.compile(r"^[+-]?\d+(?:\.\d+)?$")


@dataclass
class Config:
    w_critical: float = 4.0
    w_content: float = 2.0
    w_function: float = 0.25
    # substitution cost for two unequal tokens
    sub_mode: str = "max"            # "max" | "avg" | "sum"
    # field weight when reference field differs from template
    w_field_changed: float = 3.0
    w_field_same: float = 1.0
    # reference label not present in template_content -> treat as changed
    new_ref_field_changed: bool = True
    # penalty book-keeping for our labels not in reference / stray unlabelled content
    w_unexpected_field: float = 1.0
    # RES weighting
    w_findings: float = 0.65
    w_impression: float = 0.35


DEFAULT = Config()


def token_weight(tok: str, cfg: Config = DEFAULT) -> float:
    if _NUM_RE.match(tok):
        return cfg.w_critical
    if tok in CRITICAL_WORDS:
        return cfg.w_critical
    if tok in FUNCTION_WORDS:
        return cfg.w_function
    return cfg.w_content


# --------------------------------------------------------------------------------------
# Normalization  ->  token list
# --------------------------------------------------------------------------------------

_LIST_MARKER_RE = re.compile(r"^[ \t]*(?:\d+[.)]|[-*•])[ \t]+", re.MULTILINE)
_UNIT_SUBS = [
    (re.compile(r"\bmillimet(?:er|re)s?\b"), "mm"),
    (re.compile(r"\bcentimet(?:er|re)s?\b"), "cm"),
    (re.compile(r"\bmillilit(?:er|re)s?\b"), "ml"),
]
_LETTER_LETTER_HYPHEN = re.compile(r"(?<=[a-z])-(?=[a-z])")
_LETTER_NUM = re.compile(r"(?<=[a-z])(?=\d)")
_NUM_LETTER = re.compile(r"(?<=\d)(?=[a-z])")
_KEEP_SIGNED = re.compile(r"(?:(?<=\s)|^)([+-])(?=\d)")
_PUNCT = re.compile(r"[^\w\s]")


def normalize(text: str) -> List[str]:
    if not text:
        return []
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = _LIST_MARKER_RE.sub("", text)
    text = text.replace("\n", " ")
    for rx, rep in _UNIT_SUBS:
        text = rx.sub(rep, text)
    text = _LETTER_LETTER_HYPHEN.sub("", text)          # air-space -> airspace
    text = _LETTER_NUM.sub(" ", text)                   # 5mm -> 5 mm
    text = _NUM_LETTER.sub(" ", text)
    # protect signed measurements with a sentinel, strip other punctuation, restore
    text = _KEEP_SIGNED.sub(lambda m: "\x00" if m.group(1) == "-" else "\x01", text)
    text = _PUNCT.sub(" ", text)
    text = text.replace("\x00", "-").replace("\x01", "+")
    toks = text.split()
    out = []
    for t in toks:
        if _NUM_RE.match(t):
            out.append(t)
        else:
            t = t.strip("+-")
            if t:
                out.append(t)
    return out


# --------------------------------------------------------------------------------------
# Report parsing  ->  {label: text}, impression, unlabelled bucket
# --------------------------------------------------------------------------------------

# a "label" line: short phrase (any case) then a colon at line start.
# templates use both "BONES:" and "Bones:" / "Hip Joints:"; references always UPPERCASE them.
_LABEL_RE = re.compile(r"^[ \t]*([A-Za-z][A-Za-z0-9 /,()&'\-]{0,40}):[ \t]*(\S.*|)$")


def _is_label(m) -> bool:
    """Guard against sentence lines that happen to contain an early colon."""
    lbl = m.group(1).strip()
    return "." not in lbl and len(lbl.split()) <= 7


@dataclass
class ParsedReport:
    fields: "Dict[str, str]" = field(default_factory=dict)   # label -> text (in order)
    order: List[str] = field(default_factory=list)
    unlabelled: str = ""                                     # stray FINDINGS content
    impression: str = ""


def _split_sections(report: str) -> Tuple[str, str]:
    """Return (findings_block, impression_block)."""
    lines = report.splitlines()
    fi = ii = None
    for i, ln in enumerate(lines):
        s = ln.strip().upper()
        if fi is None and s.startswith("FINDINGS:"):
            fi = i
        elif s.startswith("IMPRESSION:"):
            ii = i
            break
    if fi is None:
        fi = -1
    findings = "\n".join(lines[fi + 1: ii if ii is not None else len(lines)])
    # keep any text after "FINDINGS:" on the same line
    if fi >= 0:
        head = lines[fi].split(":", 1)[1] if ":" in lines[fi] else ""
        findings = (head + "\n" + findings).strip("\n")
    impression = ""
    if ii is not None:
        imp_head = lines[ii].split(":", 1)[1] if ":" in lines[ii] else ""
        impression = (imp_head + "\n" + "\n".join(lines[ii + 1:])).strip()
    return findings, impression


def parse_report(report: str) -> ParsedReport:
    findings_block, impression = _split_sections(report or "")
    pr = ParsedReport(impression=impression)
    current = None
    for raw in findings_block.splitlines():
        m = _LABEL_RE.match(raw)
        if m and _is_label(m):
            label = re.sub(r"\s+", " ", m.group(1).strip()).upper()
            current = label
            if label not in pr.fields:
                pr.fields[label] = m.group(2).strip()
                pr.order.append(label)
            else:  # duplicate label - append
                pr.fields[label] += " " + m.group(2).strip()
            continue
        if raw.strip() == "":
            current = None            # blank line ends a field; trailing text -> unlabelled
            continue
        if current is None:
            pr.unlabelled += (" " if pr.unlabelled else "") + raw.strip()
        else:
            pr.fields[current] += (" " if pr.fields[current] else "") + raw.strip()
    return pr


# --------------------------------------------------------------------------------------
# Weighted ordered Levenshtein
# --------------------------------------------------------------------------------------

def _weights(toks: List[str], cfg: Config) -> List[float]:
    return [token_weight(t, cfg) for t in toks]


def weighted_word_edit(ref: List[str], sub: List[str], cfg: Config = DEFAULT) -> float:
    """Return per-field score in [0, 1]  (weighted edit cost / max total weight, capped)."""
    rw = _weights(ref, cfg)
    sw = _weights(sub, cfg)
    tot_ref, tot_sub = sum(rw), sum(sw)
    denom = max(tot_ref, tot_sub)
    if denom == 0:
        return 0.0
    n, m = len(ref), len(sub)
    # dp over rows
    prev = [0.0] * (m + 1)
    for j in range(1, m + 1):
        prev[j] = prev[j - 1] + sw[j - 1]
    for i in range(1, n + 1):
        cur = [0.0] * (m + 1)
        cur[0] = prev[0] + rw[i - 1]
        ri = ref[i - 1]
        for j in range(1, m + 1):
            if ri == sub[j - 1]:
                sub_cost = 0.0
            elif cfg.sub_mode == "avg":
                sub_cost = (rw[i - 1] + sw[j - 1]) / 2
            elif cfg.sub_mode == "sum":
                sub_cost = rw[i - 1] + sw[j - 1]
            else:
                sub_cost = max(rw[i - 1], sw[j - 1])
            cur[j] = min(
                prev[j] + rw[i - 1],        # delete ref token
                cur[j - 1] + sw[j - 1],     # insert sub token
                prev[j - 1] + sub_cost,     # substitute
            )
        prev = cur
    return min(prev[m] / denom, 1.0)


# --------------------------------------------------------------------------------------
# Case scoring
# --------------------------------------------------------------------------------------

def _norm_key(text: str) -> Tuple[str, ...]:
    return tuple(normalize(text))


def score_findings(pred: ParsedReport, ref: ParsedReport, tmpl: ParsedReport,
                   cfg: Config = DEFAULT) -> Tuple[float, list]:
    num = 0.0
    den = 0.0
    detail = []
    ref_labels = set(ref.order)

    for label in ref.order:
        ref_txt = ref.fields.get(label, "")
        tmpl_txt = tmpl.fields.get(label, None)
        if tmpl_txt is None:
            fw = cfg.w_field_changed if cfg.new_ref_field_changed else cfg.w_field_same
        else:
            fw = cfg.w_field_same if _norm_key(ref_txt) == _norm_key(tmpl_txt) else cfg.w_field_changed
        pred_txt = pred.fields.get(label, "")
        s = weighted_word_edit(normalize(ref_txt), normalize(pred_txt), cfg)
        num += fw * s
        den += fw
        detail.append((label, round(fw, 2), round(s, 4), label not in pred.fields))

    # reference unlabelled content -> pseudo-field, weight = changed (it is never in template)
    if _norm_key(ref.unlabelled):
        fw = cfg.w_field_changed
        s = weighted_word_edit(normalize(ref.unlabelled), normalize(pred.unlabelled), cfg)
        num += fw * s
        den += fw
        detail.append(("__UNLABELLED__", fw, round(s, 4), False))
    elif _norm_key(pred.unlabelled):
        num += cfg.w_unexpected_field * 1.0
        den += cfg.w_unexpected_field
        detail.append(("__UNLABELLED_EXTRA__", cfg.w_unexpected_field, 1.0, False))

    # our labels not expected by reference -> extra-content penalty
    for label in pred.order:
        if label not in ref_labels and _norm_key(pred.fields.get(label, "")):
            num += cfg.w_unexpected_field * 1.0
            den += cfg.w_unexpected_field
            detail.append((f"__EXTRA__:{label}", cfg.w_unexpected_field, 1.0, False))

    F = num / den if den else 0.0
    return F, detail


def score_case(pred_report: str, ref_report: str, template_content: str,
               cfg: Config = DEFAULT) -> dict:
    pred = parse_report(pred_report)
    ref = parse_report(ref_report)
    tmpl = parse_report(template_content)

    F, detail = score_findings(pred, ref, tmpl, cfg)
    I = weighted_word_edit(normalize(ref.impression), normalize(pred.impression), cfg)
    res = cfg.w_findings * F + cfg.w_impression * I
    return {"res": res, "F": F, "I": I, "fields": detail}


def leaderboard_res(triples, cfg: Config = DEFAULT) -> float:
    if not triples:
        return 0.0
    return sum(score_case(p, r, t, cfg)["res"] for p, r, t in triples) / len(triples)


# --------------------------------------------------------------------------------------
# self-test
# --------------------------------------------------------------------------------------


In [ ]:
# ===== prompt + house style (prompt.py) =====
"""Prompt construction: system guide + retrieved few-shot exemplars + target case."""
from __future__ import annotations

from typing import List

HOUSE_STYLE = '# House style (mined from 636 reference reports)\n\nFORM\n- Output exactly: `FINDINGS:` then the template\'s fields, one blank line, then `IMPRESSION:`.\n- Reproduce EVERY template field label, in template order, as `LABEL:` in UPPERCASE (even if the\n  template shows it in Title Case). Never drop a label. Never invent a new label.\n- One blank line between top-level anatomic sections.\n\nEDIT THE TEMPLATE - do not rewrite it\n- Start from the template text. For every field the dictation does NOT mention, copy the template\n  sentence UNCHANGED.\n- For a field the dictation DOES mention: state the finding first as a full sentence\n  ("There is ...", "There are ...", "<Finding> is present.", "<Finding> is noted."), then keep\n  the still-true part of the template sentence ("No acute fracture.").\n- Never leave a field reading as fully normal if the dictation names a finding for it - even a\n  two-word dictation ("mild degen changes") is a finding you MUST incorporate.\n\nROUTING (a finding under the wrong label is penalised twice)\n- Route each finding to the field whose anatomy it names:\n  arthroplasty / prosthesis / hardware / screws -> the JOINTS (or equivalent) field.\n  soft-tissue swelling / calcification / foreign body -> the SOFT TISSUES field.\n  osteophytes / spurs / fracture / lucency / sclerosis -> the BONES / VERTEBRAE field.\n  disc / canal / foramen -> the disc-level field.\n- Use the wording of THIS study\'s body part (a foot study says "metatarsophalangeal", a hand\n  study says "metacarpophalangeal").\n\nFIDELITY\n- Every finding in the dictation must appear exactly once. Do not omit any.\n- Add NOTHING that is not in the dictation or the template - no extra measurements, no extra\n  negatives, no comparisons, no recommendations, no "clinical correlation" unless dictated.\n- Expand shorthand: degen -> degenerative, fx -> fracture, c/w -> consistent with,\n  DJD -> degenerative joint disease, si joint -> sacroiliac joint. Fix obvious typos.\n- Resolve placeholders (`[generic]`, `[left/right]`, `[_laterality_]`) from the study description.\n- Keep negation, laterality (left/right/bilateral), severity (mild/moderate/severe) and every\n  measurement + unit exactly as dictated.\n\nIMPRESSION\n- One numbered line per abnormal finding, most significant first, phrased as a full localised\n  statement with severity ("Mild degenerative changes of the pelvis.", not "Degenerative changes.").\n- Then a final line stating the residual normal status ("No acute fracture or dislocation.",\n  "No acute cardiopulmonary abnormality.", "No acute intracranial abnormality.").\n- If the dictation is entirely normal, the IMPRESSION is the template\'s normal summary line.\n- Introduce no finding in the IMPRESSION that is not in FINDINGS.\n\nTHE EXAMPLES ARE DIFFERENT PATIENTS\n- Use them only to learn wording and section routing. NEVER copy a finding, measurement, or\n  non-normal sentence from an example into your report.\n'

SYSTEM = f"""You are a radiology report editor. You are given a normal report TEMPLATE and a \
radiologist's telegraphic DICTATION. Produce the final report by merging the dictation's findings \
into the template.

You are scored by an edit-distance metric (RES) that rewards reproducing the reference report's \
exact wording, field routing and word order, and heavily penalises invented content, omitted \
findings, wrong-field routing and unnecessary rewriting of normal text.

{HOUSE_STYLE}

Output ONLY the final report, starting with `FINDINGS:` and containing `IMPRESSION:`. No preamble, \
no commentary, no code fences."""

_RULES = (
    "Reproduce every template field label in order (UPPERCASE). Copy every field the dictation "
    "does not mention, unchanged. Merge each dictated finding into the field whose anatomy it "
    "names. Omit nothing from the dictation; add nothing not in the dictation or template. Do not "
    "copy anything from the examples."
)


def _case_block(r: dict, with_report: bool) -> str:
    parts = [
        f"MODALITY: {r['modality']}   BODY PART: {r['body_part']}   STUDY: {r['study_description']}",
        f"PATIENT: {r['patient_age_band']} {r['patient_sex']}",
        "",
        "TEMPLATE:",
        r["template_content"].strip(),
        "",
        "DICTATION:",
        r["dictation"].strip(),
    ]
    block = "\n".join(parts)
    block += "\n\nFINAL REPORT:\n" + (r["report"].strip() if with_report else "")
    return block


def build_messages(case: dict, exemplars: List[dict]):
    """Return (system, messages) for client.messages.create."""
    chunks = [
        "## HOUSE-STYLE EXAMPLES (different patients — for wording and routing only, never copy "
        "their findings)\n"
    ]
    for i, ex in enumerate(exemplars, 1):
        chunks.append(f"----- EXAMPLE {i} -----\n{_case_block(ex, with_report=True)}")
    chunks.append(
        "## YOUR CASE\n\n" + _case_block(case, with_report=False).rstrip()
        + "\n\nWrite the FINAL REPORT now. " + _RULES
    )
    return SYSTEM, [{"role": "user", "content": "\n\n".join(chunks)}]

In [ ]:
# ===== retrieval (exemplars.py) =====
"""Retrieve the k most relevant train examples for a given case.

Ranking signal (no embeddings needed - the templates are near-duplicates within a
modality x body_part bucket):

    same (modality, body_part)          strong bonus
    template_content similarity         primary (matching template => matching field
                                        structure and house rephrasing conventions)
    study_description similarity        tie-breaker
    similar dictation length            mild bonus (comparable abnormality load)
"""
from __future__ import annotations

from difflib import SequenceMatcher
from typing import List



def _ratio(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b, autojunk=False).ratio()


def _norm_join(s: str) -> str:
    return " ".join(normalize(s))


def score_pair(case: dict, cand: dict) -> float:
    s = 0.0
    if case["modality"] == cand["modality"]:
        s += 2.0
    if case["body_part"] == cand["body_part"]:
        s += 3.0
    tmpl_sim = _ratio(_norm_join(case["template_content"]), _norm_join(cand["template_content"]))
    s += 6.0 * tmpl_sim
    if tmpl_sim > 0.98:
        s += 4.0                       # essentially the same template
    s += 1.5 * _ratio(case["study_description"].lower(), cand["study_description"].lower())
    la, lb = len(case["dictation"]), len(cand["dictation"])
    s += 0.5 * (min(la, lb) / max(la, lb, 1))
    return s


def retrieve(case: dict, pool: List[dict], k: int = 4) -> List[dict]:
    ranked = sorted(pool, key=lambda c: score_pair(case, c), reverse=True)
    return ranked[:k]

In [ ]:
# ===== post-processor (postprocess.py) =====
"""Deterministic post-processing: force the model output onto the template's exact skeleton.

Guarantees, regardless of what the model returned:
  - starts with `FINDINGS:`, contains `IMPRESSION:`
  - every template field label reproduced verbatim, in template order
  - template blank-line layout preserved
  - a field the model did not fill falls back to the template's normal sentence
  - the model's unlabelled trailing findings (if any) kept just before IMPRESSION
"""
from __future__ import annotations

import re



def _norm_label(lbl: str) -> str:
    return re.sub(r"\s+", " ", lbl.strip()).upper()


def reskeleton(template_content: str, model_report: str) -> str:
    tmpl_lines = template_content.splitlines()
    mp = parse_report(model_report)
    model_fields = {_norm_label(k): v.strip() for k, v in mp.fields.items()}

    # locate FINDINGS: / IMPRESSION: in the template
    f_idx = i_idx = None
    for i, ln in enumerate(tmpl_lines):
        s = ln.strip().upper()
        if f_idx is None and s.startswith("FINDINGS:"):
            f_idx = i
        elif s.startswith("IMPRESSION:"):
            i_idx = i
            break
    if f_idx is None:
        # template has no explicit FINDINGS header - just trust the model output
        return model_report.strip()

    out = ["FINDINGS:"]
    body_end = i_idx if i_idx is not None else len(tmpl_lines)
    for ln in tmpl_lines[f_idx + 1: body_end]:
        m = _LABEL_RE.match(ln)
        if m and _is_label(m):
            label = _norm_label(m.group(1))          # references always UPPERCASE labels
            tmpl_text = m.group(2).strip()
            new_text = model_fields.get(label, tmpl_text)
            out.append(f"{label}: {new_text}".rstrip())
        else:
            out.append(ln.rstrip())            # blank lines / stray template lines

    # model's unlabelled trailing findings, if any
    if mp.unlabelled.strip():
        if out and out[-1].strip():
            out.append("")
        out.append(mp.unlabelled.strip())

    # IMPRESSION
    imp = mp.impression.strip()
    if not imp and i_idx is not None:
        imp = "\n".join(tmpl_lines[i_idx + 1:]).strip()
    out += ["", "IMPRESSION:", imp]

    text = "\n".join(out)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

In [ ]:
# Radiology Reporting Harness — retrieval + few-shot + skeleton pipeline (open model, Kaggle GPU)
#
# HOW TO USE
#   1. + Add Input -> Competitions -> the radiology reporting harness competition
#   2. Accelerator (right panel): GPU T4 x2
#   3. Internet: ON  (first run downloads the model from Hugging Face)
#   4. Set MODE in the CONFIG cell.  "cv" = measure local RES;  "submit" = write submission.csv
#   5. Run -> "Restart & Run All"  (NOT plain "Run All" — a stale kernel keeps dead
#      model copies in GPU memory and the load cell then OOMs)
#   6. Save Version the moment it finishes.  Outputs -> /kaggle/working/
#
# Backend: transformers + bitsandbytes 4-bit (matches Kaggle's CUDA 12 image; vLLM's current
# build needs CUDA 13). No pip -U — that breaks Kaggle's pinned deps.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, subprocess, sys
print("torch", torch.__version__, "| cuda", torch.version.cuda,
      "| gpu", torch.cuda.is_available(), torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU — set Accelerator to GPU T4 x2 and restart the session"
try:
    import bitsandbytes  # noqa
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "bitsandbytes"], check=True)
    import bitsandbytes  # noqa
print("bitsandbytes", bitsandbytes.__version__)

In [ ]:
# ---------------- CONFIG ----------------
MODE       = "cv"        # "cv" | "submit"
CV_FOLD    = 0
CV_LIMIT   = 100         # cap CV cases (None = whole fold)
K          = 2           # few-shot exemplars per case

MODEL      = "Qwen/Qwen2.5-14B-Instruct"    # fallback if it OOMs at load: "Qwen/Qwen2.5-7B-Instruct"
LOAD_4BIT  = True
MAX_INPUT_TOKENS = 9000
MAX_NEW_TOKENS   = 1500   # long MRI reports need this; 1024 truncated them
BATCH_SIZE = 4           # length-bucketed; generation cell auto-halves any batch that OOMs
DEVICE_MAP = "auto"      # spread the 7B across both T4s for KV-cache headroom

WORK       = "/kaggle/working"   # DATA_DIR auto-detected under /kaggle/input below
# ---------------------------------------

In [ ]:
import json, statistics, re, glob, os
from collections import defaultdict
import pandas as pd

_cands = glob.glob("/kaggle/input/**/train.csv", recursive=True)
assert _cands, "train.csv not found under /kaggle/input — use '+ Add Input' to attach the competition data"
DATA_DIR = os.path.dirname(_cands[0])
print("DATA_DIR =", DATA_DIR, "->", sorted(os.listdir(DATA_DIR)))

train = pd.read_csv(f"{DATA_DIR}/train.csv").fillna("").to_dict("records")
test  = pd.read_csv(f"{DATA_DIR}/test.csv").fillna("").to_dict("records")
print(f"train {len(train)}  test {len(test)}")

def make_folds(rows, k=5):
    strata = defaultdict(list)
    for r in rows:
        strata[(r["modality"], r["body_part"])].append(r["case_id"])
    fold, n = {}, 0
    for key in sorted(strata):
        for cid in sorted(strata[key]):
            fold[cid] = n % k
            n += 1
    return fold

folds = make_folds(train, 5)

if MODE == "cv":
    targets = [r for r in train if folds[r["case_id"]] == CV_FOLD]
    if CV_LIMIT:
        targets = targets[:CV_LIMIT]
    def pool_for(c):
        return [r for r in train if folds[r["case_id"]] != folds[c["case_id"]]]
else:
    targets = test
    def pool_for(c):
        return train
print(f"MODE={MODE}  targets={len(targets)}")

In [ ]:
# Build prompts (retrieve() / build_messages() come from the inlined pipeline cells above)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def make_prompt(case, k):
    ex = retrieve(case, pool_for(case), k=k)
    system, messages = build_messages(case, ex)
    chat = [{"role": "system", "content": system}] + messages
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    return text, [e["case_id"] for e in ex]

prompts, meta = [], []
for c in targets:
    p, exids = make_prompt(c, K)
    prompts.append(p)
    meta.append({"case_id": c["case_id"], "exemplar_ids": exids})

ntok = [len(tok(p).input_ids) for p in prompts]
print(f"prompt tokens: median {int(statistics.median(ntok))}  max {max(ntok)}  (cap {MAX_INPUT_TOKENS})")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import gc

for _n in ("model", "out", "enc"):          # free any leftovers from an earlier run
    if _n in dir():
        del globals()[_n]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for _d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(_d)
        print(f"  cuda:{_d}  free {free/1e9:.1f} / {total/1e9:.1f} GB")

kw = dict(torch_dtype=torch.float16, device_map=DEVICE_MAP)
if LOAD_4BIT:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(MODEL, **kw).eval()
print("loaded", MODEL, "| device map", set(model.hf_device_map.values()) if hasattr(model, "hf_device_map") else "n/a")

In [ ]:
import time, gc

def _decode(batch, max_new):
    enc = tok(batch, return_tensors="pt", padding=True, truncation=True,
              max_length=MAX_INPUT_TOKENS).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             num_beams=1, pad_token_id=tok.pad_token_id)
    n_in = enc.input_ids.shape[1]
    res = [tok.decode(out[j][n_in:], skip_special_tokens=True).strip() for j in range(len(batch))]
    del enc, out
    return res

def gen_batch(batch, max_new=MAX_NEW_TOKENS):
    """Greedy-decode a list of prompts; recursively halve on CUDA OOM."""
    try:
        return _decode(batch, max_new)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(batch) == 1:
            return None            # caller handles (salvage pass)
        mid = len(batch) // 2
        print(f"   OOM on batch of {len(batch)} -> splitting")
        left = gen_batch(batch[:mid], max_new); right = gen_batch(batch[mid:], max_new)
        return [x for x in (left or [None] * mid)] + [x for x in (right or [None] * (len(batch) - mid))]

def looks_bad(r):
    return (r is None) or (len(r.strip()) < 40) or ("IMPRESSION" not in r.upper())

# length-bucket so each batch has similar padding (less OOM, faster, cleaner output)
order = sorted(range(len(prompts)), key=lambda i: len(tok(prompts[i]).input_ids))
raw = [None] * len(prompts)
t0 = time.time()
for b in range(0, len(order), BATCH_SIZE):
    idx = order[b:b + BATCH_SIZE]
    res = gen_batch([prompts[i] for i in idx])
    for i, r in zip(idx, res):
        raw[i] = r
    done = min(b + BATCH_SIZE, len(order))
    print(f"  {done}/{len(order)}   {(time.time()-t0)/done:.1f}s/case")
print(f"main pass done in {(time.time()-t0)/60:.1f} min")

# salvage: retry bad/empty cases individually with fewer, shorter exemplars
bad = [i for i in range(len(raw)) if looks_bad(raw[i])]
print(f"salvage pass: {len(bad)} cases")
for i in bad:
    for kk in (2, 1):
        p, _ = make_prompt(targets[i], kk)
        r = gen_batch([p])
        r = r[0] if r else None
        if not looks_bad(r):
            raw[i] = r
            break
    if looks_bad(raw[i]):
        raw[i] = raw[i] or ""      # give up -> skeleton falls back to template
print(f"still bad after salvage: {sum(looks_bad(raw[i]) for i in range(len(raw)))}")
raw = ["" if r is None else r for r in raw]
assert len(raw) == len(prompts)

In [ ]:
preds = [reskeleton(c["template_content"], r) for c, r in zip(targets, raw)]

if MODE == "cv":
    b0  = [score_case(c["template_content"], c["report"], c["template_content"])["res"] for c in targets]
    sc  = [score_case(p, c["report"], c["template_content"]) for c, p in zip(targets, preds)]
    res = [s["res"] for s in sc]

    print(f"\n=== CV fold {CV_FOLD}  n={len(res)}  model={MODEL} ===")
    print(f"B0 (emit template)  mean RES = {statistics.mean(b0):.4f}")
    print(f"pipeline            mean RES = {statistics.mean(res):.4f}   median = {statistics.median(res):.4f}")
    print(f"                    F = {statistics.mean(s['F'] for s in sc):.3f}   I = {statistics.mean(s['I'] for s in sc):.3f}")
    bucket = defaultdict(list)
    for c, s in zip(targets, sc):
        bucket[c["modality"]].append(s["res"])
    for m in sorted(bucket):
        print(f"    {m:5s} n={len(bucket[m]):3d}  mean RES = {statistics.mean(bucket[m]):.4f}")

    out = []
    for c, p, r, s, m in zip(targets, preds, raw, sc, meta):
        out.append({"case_id": c["case_id"], "modality": c["modality"], "body_part": c["body_part"],
                    "res": s["res"], "F": s["F"], "I": s["I"],
                    "dictation": c["dictation"], "template_content": c["template_content"],
                    "gold": c["report"], "pred": p, "raw": r, "exemplar_ids": m["exemplar_ids"]})
    out.sort(key=lambda x: -x["res"])
    json.dump(out, open(f"{WORK}/cv_results.json", "w"), ensure_ascii=False, indent=1)
    with open(f"{WORK}/cv_summary.txt", "w") as f:
        f.write(f"fold {CV_FOLD} n={len(res)} model={MODEL}\nB0 {statistics.mean(b0):.4f}\n"
                f"pipeline {statistics.mean(res):.4f} median {statistics.median(res):.4f}\n\nworst 15:\n")
        for x in out[:15]:
            f.write(f"  {x['res']:.3f}  {x['modality']:5s} {x['body_part']}\n")
    print(f"\nwrote {WORK}/cv_results.json  and  {WORK}/cv_summary.txt")

    # ---- inline diagnostics (screenshot this — no download needed) ----
    print("\n" + "=" * 70 + "\nWORST 6 CASES\n" + "=" * 70)
    for x in out[:6]:
        print(f"\n### RES {x['res']:.3f}  ({x['modality']} {x['body_part']})  F={x['F']:.2f} I={x['I']:.2f}")
        print("- DICTATION:", " ".join(x["dictation"].split())[:300])
        print("- GOLD:\n" + x["gold"][:700])
        print("- PRED:\n" + x["pred"][:700])
    print("\n" + "=" * 70 + "\nMEDIAN-DIFFICULTY CASE (for style check)\n" + "=" * 70)
    mid = out[len(out) // 2]
    print(f"RES {mid['res']:.3f}  ({mid['modality']} {mid['body_part']})")
    print("- DICTATION:", " ".join(mid["dictation"].split())[:300])
    print("- GOLD:\n" + mid["gold"][:700])
    print("- PRED:\n" + mid["pred"][:700])
else:
    sub = pd.DataFrame({"case_id": [c["case_id"] for c in targets], "report": preds})
    assert sub["case_id"].is_unique and len(sub) == len(test), "submission shape wrong"
    sub.to_csv(f"{WORK}/submission.csv", index=False)
    json.dump(meta, open(f"{WORK}/submission_meta.json", "w"), indent=1)
    print(f"wrote {WORK}/submission.csv  rows={len(sub)}")
    print(f"report chars: median {int(sub.report.str.len().median())}  max {int(sub.report.str.len().max())}")
    print("--- example ---\n" + sub.iloc[0].report[:700])